In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
branch_table = dbutils.widgets.get("branch_table")
office_table = dbutils.widgets.get("office_table")
clientepisodefsall_table = dbutils.widgets.get("clientepisodefsall_table")
payorsources_table = dbutils.widgets.get("payorsources_table")
payor_table = dbutils.widgets.get("payor_table")
client_table = dbutils.widgets.get("client_table")
clientepisodesall_table = dbutils.widgets.get("clientepisodesall_table")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW insurance_src AS
SELECT 
  CAST(ReportingDate AS DATE) AS ReportingDate,
  CAST(FacilityCode AS STRING) AS FacilityCode,
  CAST(AcctNbr AS STRING) AS AcctNbr,
  CAST(InsRank AS INT) AS InsRank,
  CAST(ActiveIns AS INT) AS ActiveIns,
  CAST(InsCode AS STRING) AS InsCode,
  NULL AS InsEDIID,
  NULL AS InsEID,
  NULL AS InsGroupID,
  CAST(InsPolicy AS STRING) AS InsPolicy,
  NULL AS InsPreCertNbr,
  CAST(InsBalance AS DOUBLE) AS InsBalance,
  CAST(InsPymt AS DOUBLE) AS InsPymt,
  NULL AS InsAdj,
  CAST(InsName AS STRING) AS InsName,
  CAST(InsAddr1 AS STRING) AS InsAddr1,
  NULL AS InsAddr2, 
  CAST(InsCity AS STRING) AS InsCity,
  CAST(InsState AS STRING) AS InsState,
  CAST(InsZip AS STRING) AS InsZip,
  NULL AS InsCountry,
  NULL AS InsProvince,
  NULL AS InsPhone,
  NULL AS InsEmail,
  CAST(InsSubFName AS STRING) AS InsSubFName,
  CAST(InsSubMName AS STRING) AS InsSubMName,
  CAST(InsSubLName AS STRING) AS InsSubLName,
  NULL AS InsSubSuffix,
  CAST(InsSubAddr1 AS STRING) AS InsSubAddr1,
  NULL AS InsSubAddr2,
  CAST(InsSubCity AS STRING) AS InsSubCity,
  CAST(InsSubState AS STRING) AS InsSubState,
  CAST(InsSubZip AS STRING) AS InsSubZip,
  NULL AS InsSubCountry,
  NULL AS InsSubProvince,
  CAST(InsSubPhone AS STRING) AS InsSubPhone,
  CAST(InsSubDOB AS STRING) AS InsSubDOB,
  NULL AS InsSubSSN,
  CAST(InsSubGender AS STRING) AS InsSubGender,
  NULL AS InsSubRelation,
  CAST(InitialBillDate AS STRING) AS InitialBillDate,
  NULL AS LastBillDate,
  NULL AS LastBillSubmitDate,
  NULL AS LastBillType,
  NULL AS LastMediaType,
  NULL AS InsStatusCode,
  NULL AS InsStatusDate,
  CAST(SourceSystemKey AS INT) AS SourceSystemKey
FROM (
  WITH 
  insurance_cte AS (
    SELECT  
    CAST('{fetch_date}' AS DATE) AS ReportingDate,
    CASE  
        WHEN b.branch_code RLIKE '[A-Za-z]' THEN ofc.OfficeNumber  
        ELSE b.branch_code  
    END AS FacilityCode, 
    bi_h.i_id AS AcctNbr, 
    '1' AS InsRank,
    '1' AS ActiveIns,
    ps.ps_id AS InsCode,
    NULL AS InsGroupID,
    COALESCE(
        NULLIF(TRIM(REPLACE(REPLACE(REPLACE(cefs.cefs_policyno, CHAR(13), ''), CHAR(10), ''), CHAR(9), '')), ''),
        cefs.cefs_medicareno
    ) AS InsPolicy,
    bi_h.i_charge AS InsBalance,
    bi_h.i_balance AS InsPymt,
    ps.ps_desc AS InsName,
    pyr.Address1 AS InsAddr1,  -- Added from payor table
    pyr.City AS InsCity ,
    pyr.State AS InsState, 
    pyr.Zip AS InsZip,
    c.FirstName AS InsSubFName,
    c.MI AS InsSubMName,
    c.LastName AS InsSubLName,
    c.Address AS InsSubAddr1,
    c.City AS InsSubCity,
    c.State AS InsSubState,
    c.ZipCode AS InsSubZip,  
    -- c.,
    CONCAT(c.AreaCode, REGEXP_REPLACE(c.PhoneNumber, '[^0-9]', '')) AS InsSubPhone,
    date_format(c.DateOfBirth, 'yyyyMMdd') AS InsSubDOB,
    CASE 
        WHEN c.Gender IN ('Male', 'Female', 'MALE', 'FEMALE') 
        THEN SUBSTRING(c.Gender, 1, 1) 
        ELSE NULL 
    END AS InsSubGender,  
    date_format(bi_h.i_postdate, 'yyyy-MM-dd') AS InitialBillDate, 
    '6' AS SourceSystemKey 
    FROM {source_table} bi_h 
    JOIN {branch_table} b  
        ON bi_h.i_branchcode = b.branch_code 
    LEFT JOIN {office_table} ofc 
        ON ofc.OfficeAbbreviation = b.branch_code 
    JOIN {clientepisodefsall_table} cefs
        ON cefs.cefs_id = bi_h.i_cefsid
    LEFT JOIN {payorsources_table} ps
        ON ps.ps_id = cefs.cefs_psid
    LEFT JOIN prd_bronze_raw.mart.payor pyr  -- ADDED THIS JOIN
        ON CAST(pyr.PayorID AS STRING) = CAST(ps.ps_id AS STRING)
    JOIN {clientepisodesall_table} epi
        ON epi.epi_id = cefs.cefs_epiid
    JOIN {client_table} c --there are  duplicates in this source table
        ON c.pa_id = epi.epi_paid
    -- JOIN (
    --     SELECT *,
    --         ROW_NUMBER() OVER (PARTITION BY pa_id ORDER BY ClientID) as rn
    --     FROM {client_table}
    --     WHERE Deleted = false
    -- ) c
    --     ON c.pa_id = epi.epi_paid
    --     AND c.rn = 1
    WHERE bi_h.i_Balance <> 0
  ),
  insurance_clean AS (
    SELECT *,
    row_number() OVER (PARTITION BY AcctNbr ORDER BY AcctNbr ) AS rn
    FROM insurance_cte
  )
  SELECT 
    ReportingDate,
    FacilityCode,
    AcctNbr,
    InsRank,
    ActiveIns,
    InsCode,
    InsGroupID,
    InsPolicy,
    InsBalance,
    InsPymt,
    InsName,
    InsAddr1,
    InsCity,
    InsState,
    InsZip,
    InsSubFName,
    InsSubMName,
    InsSubLName,
    InsSubAddr1,
    InsSubCity,
    InsSubState,
    InsSubZip,
    InsSubPhone,
    InsSubDOB,
    InsSubGender,
    InitialBillDate,
    SourceSystemKey
  FROM insurance_clean
  WHERE rn=1
)
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING insurance_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 6

WHEN MATCHED THEN
UPDATE SET
    tgt.FacilityCode = src.FacilityCode,
    tgt.InsRank = src.InsRank,
    tgt.ActiveIns = src.ActiveIns,
    tgt.InsCode = src.InsCode,
    tgt.InsEDIID = src.InsEDIID,
    tgt.InsEID = src.InsEID,
    tgt.InsGroupID = src.InsGroupID,
    tgt.InsPolicy = src.InsPolicy,
    tgt.InsPreCertNbr = src.InsPreCertNbr,
    tgt.InsBalance = src.InsBalance,
    tgt.InsPymt = src.InsPymt,
    tgt.InsAdj = src.InsAdj,
    tgt.InsName = src.InsName,
    tgt.InsAddr1 = src.InsAddr1,
    tgt.InsAddr2 = src.InsAddr2,
    tgt.InsCity = src.InsCity,
    tgt.InsState = src.InsState,
    tgt.InsZip = src.InsZip,
    tgt.InsCountry = src.InsCountry,
    tgt.InsProvince = src.InsProvince,
    tgt.InsPhone = src.InsPhone,
    tgt.InsEmail = src.InsEmail,
    tgt.InsSubFName = src.InsSubFName,
    tgt.InsSubMName = src.InsSubMName,
    tgt.InsSubLName = src.InsSubLName,
    tgt.InsSubSuffix = src.InsSubSuffix,
    tgt.InsSubAddr1 = src.InsSubAddr1,
    tgt.InsSubAddr2 = src.InsSubAddr2,
    tgt.InsSubCity = src.InsSubCity,
    tgt.InsSubState = src.InsSubState,
    tgt.InsSubZip = src.InsSubZip,
    tgt.InsSubCountry = src.InsSubCountry,
    tgt.InsSubProvince = src.InsSubProvince,
    tgt.InsSubPhone = src.InsSubPhone,
    tgt.InsSubDOB = src.InsSubDOB,
    tgt.InsSubSSN = src.InsSubSSN,
    tgt.InsSubGender = src.InsSubGender,
    tgt.InsSubRelation = src.InsSubRelation,
    tgt.InitialBillDate = src.InitialBillDate,
    tgt.LastBillDate = src.LastBillDate,
    tgt.LastBillSubmitDate = src.LastBillSubmitDate,
    tgt.LastBillType = src.LastBillType,
    tgt.LastMediaType = src.LastMediaType,
    tgt.InsStatusCode = src.InsStatusCode,
    tgt.InsStatusDate = src.InsStatusDate,
    tgt.SourceSystemKey = src.SourceSystemKey,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    ReportingDate,
    FacilityCode,
    AcctNbr,
    InsRank,
    ActiveIns,
    InsCode,
    InsEDIID,
    InsEID,
    InsGroupID,
    InsPolicy,
    InsPreCertNbr,
    InsBalance,
    InsPymt,
    InsAdj,
    InsName,
    InsAddr1,
    InsAddr2,
    InsCity,
    InsState,
    InsZip,
    InsCountry,
    InsProvince,
    InsPhone,
    InsEmail,
    InsSubFName,
    InsSubMName,
    InsSubLName,
    InsSubSuffix,
    InsSubAddr1,
    InsSubAddr2,
    InsSubCity,
    InsSubState,
    InsSubZip,
    InsSubCountry,
    InsSubProvince,
    InsSubPhone,
    InsSubDOB,
    InsSubSSN,
    InsSubGender,
    InsSubRelation,
    InitialBillDate,
    LastBillDate,
    LastBillSubmitDate,
    LastBillType,
    LastMediaType,
    InsStatusCode,
    InsStatusDate,
    SourceSystemKey,
    _load_timestamp
)
VALUES (
    src.ReportingDate,
    src.FacilityCode,
    src.AcctNbr,
    src.InsRank,
    src.ActiveIns,
    src.InsCode,
    src.InsEDIID,
    src.InsEID,
    src.InsGroupID,
    src.InsPolicy,
    src.InsPreCertNbr,
    src.InsBalance,
    src.InsPymt,
    src.InsAdj,
    src.InsName,
    src.InsAddr1,
    src.InsAddr2,
    src.InsCity,
    src.InsState,
    src.InsZip,
    src.InsCountry,
    src.InsProvince,
    src.InsPhone,
    src.InsEmail,
    src.InsSubFName,
    src.InsSubMName,
    src.InsSubLName,
    src.InsSubSuffix,
    src.InsSubAddr1,
    src.InsSubAddr2,
    src.InsSubCity,
    src.InsSubState,
    src.InsSubZip,
    src.InsSubCountry,
    src.InsSubProvince,
    src.InsSubPhone,
    src.InsSubDOB,
    src.InsSubSSN,
    src.InsSubGender,
    src.InsSubRelation,
    src.InitialBillDate,
    src.LastBillDate,
    src.LastBillSubmitDate,
    src.LastBillType,
    src.LastMediaType,
    src.InsStatusCode,
    src.InsStatusDate,
    src.SourceSystemKey,
    current_timestamp()
)
""")
)